Supervised Training for QCGPT

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
if (root / 'qcgpt').exists():
    sys.path.insert(0, str(root))
elif (root.parent / 'qcgpt').exists():
    sys.path.insert(0, str(root.parent))


In [ ]:
import os
import time
import csv
import torch
import torch.optim as optim
from qcgpt.gates import VOCAB
from qcgpt.models.policy import CircuitPolicy
from qcgpt.training.supervised import build_simplified_dataloader, train_supervised_epoch


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vocab_size = len(VOCAB)
model = CircuitPolicy(vocab_size=vocab_size, d_model=256, n_layers=4, n_heads=4, max_spec_len=256, max_circ_len=128).to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-4)
train_loader = build_simplified_dataloader(num_samples=10000, batch_size=64, n_qubits=2, raw_max_depth=16, include_basis_states=True, n_random_states=0, num_workers=0)


In [ ]:
ts = time.strftime('%Y%m%d_%H%M%S')
out_dir = Path('model_checkpoints') / ts
out_dir.mkdir(parents=True, exist_ok=True)
prefix = 'transformer_v1'
loss_csv = out_dir / f'{prefix}_loss.csv'
with open(loss_csv, 'w', newline='') as f:
    writer = csv.writer(f); writer.writerow(['epoch','loss'])
best_loss = float('inf')
for epoch in range(1, 21):
    train_loss = train_supervised_epoch(model=model, dataloader=train_loader, optimizer=optimizer, device=device)
    print(f'[Supervised] Epoch {epoch:03d}  TrainLoss={train_loss:.4f}')
    with open(loss_csv, 'a', newline='') as f:
        writer = csv.writer(f); writer.writerow([epoch, f'{train_loss:.8f}'])
    if train_loss < best_loss:
        best_loss = train_loss
        torch.save({'model_state_dict': model.state_dict()}, out_dir / f'{prefix}_best.pt')
torch.save({'model_state_dict': model.state_dict()}, out_dir / f'{prefix}_final.pt')
print('Saved best and final under', out_dir)
